> 🚨 **[Warning] 무단 도용, 복제 및 배포 금지 안내**
>
> 저작권법에 따라 강의에 사용된 모든 저작물 (코드, 프롬프트, PDF, 실습자료 등)을  
> 무단 복제하거나 외부에 유출할 경우 **_법적 문제가 발생할 수 있습니다._**


# 📂 <font color='#1A4BC0'><b>Part 03. 프롬프트 분석</b></font>

## <font color='Darkorange'><b>[ Chapter 02 ]</b></font> Turn Analysis 분석
해당 챕터는 **주피터 노트북 실습 기반**으로 진행됩니다.  
실습 시작 전 아래 설정을 반드시 실행해주세요.



```
💡 주피터 노트북 같은 경우, session으로 관리가 됩니다.
일정 시간이 지날 동안 아무런 동작을 하지 않거나, 새로운 브라우저에서 접속한 경우 아래 라이브러리들을 다시 설치해야합니다.

다시 실행해주세요.
```

### ⚙️ <font color='#007A45'><b>[ 실습 전 ]</b></font> Part3 Chapter 02 실습 전 프로젝트 셋업
>  ✅ 아래 **실습 전 가상환경을 활성화하고, 프로젝트 셋업**을 완료한 후 본 실습을 진행해주세요.

> ⚠️ 실습 진행 중 에러가 발생하거나, 세션이 종료되어 런타임이 재시작된 경우, 이 블럭을 항상 다시 실행해주세요.


```
💡 주피터 노트북 같은 경우, session으로 관리가 됩니다.
일정 시간이 지날 동안 아무런 동작을 하지 않거나, 새로운 브라우저에서 접속한 경우 아래 라이브러리들을 다시 설치해야합니다.
```

#### 실습 진행을 위한 라이브러리 다운로드

실습을 진행하기 위해서는 각 AI서비스들의 라이브러리들을 설치해야합니다.
아래 코드 블럭을 실행해서 라이브러리를 설치해봅시다!

```
💡 앞으로 아래 블럭과 같은 코드 블럭은 해당 블럭을 클릭하신 다음 왼쪽의 실행버튼(▶️)을 클릭하거나, `shift + Enter` 단축키를 통해 실행합니다.
```

In [ ]:
# 필요한 패키지 설치 (최초 1회만)
%pip install -r requirement.txt

#### 실습 진행을 위한 API KEY 세팅

실습을 진행하기 위해서는 각 AI서비스들의 API Key를 발급 및 세팅 해야합니다.

LangSmith, Gemini, Claude, Chat GPT API Key를 모두 발급하셨다면, 아래 코드 블럭을 실행하여 API Key를 세팅해봅시다.


In [ ]:
# LangSmith & OpenAI Key 설정
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# API KEY 정보 로드
load_dotenv()

# ChatOpenAI 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")

print("✅ 환경 설정 완료!")
print(f"사용 모델: gpt-4o-mini")
response = model.invoke("안녕하세요?")
print(response.content)

#### 실습 진행을 위해 모델 호출 함수 정의 세팅

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 기본 모델 생성
models = {
    "openai": ChatOpenAI(model="gpt-4o-mini"),
    "claude": ChatAnthropic(model="claude-3-5-haiku-20241022"),
    "gemini": ChatGoogleGenerativeAI(model="gemini-2.0-flash"),
}

parser = StrOutputParser()


# 베이스 체인
def _run_base_chain(model_obj, system_prompt=None, user_input=None, **overrides):
    if overrides:
        model_obj = model_obj.with_config(**overrides)

    model_name = str(type(model_obj)).lower()

    if "claude" in model_name:
        if not user_input and system_prompt:
            user_input = system_prompt
            system_prompt = None

    messages = []
    if system_prompt:
        messages.append(("system", system_prompt))
    if user_input:
        messages.append(("user", user_input))

    if not messages:
        raise ValueError("Claude requires at least one user or system message.")

    prompt = ChatPromptTemplate.from_messages(messages)
    chain = prompt | model_obj | parser
    return chain.invoke({})


# 모델별 메인 실행 체인
def run_openai_chain(system_prompt=None, user_input=None, **kwargs):
    return _run_base_chain(models["openai"], system_prompt, user_input, **kwargs)


def run_claude_chain(system_prompt=None, user_input=None, **kwargs):
    return _run_base_chain(models["claude"], system_prompt, user_input, **kwargs)


def run_gemini_chain(system_prompt=None, user_input=None, **kwargs):
    return _run_base_chain(models["gemini"], system_prompt, user_input, **kwargs)

실행 예시를 알아봅시다.

In [ ]:
# 실행 예시
system_prompt = "You are a concise and helpful AI assistant."
user_input = "너에 대해 소개해줘!"

# OpenAI 모델 실행
print("# OpenAI Result:")
print(run_openai_chain(system_prompt=system_prompt, user_input=user_input))
print("-" * 40)

# Claude 모델 실행
print("# Claude Result:")
print(run_claude_chain(system_prompt=system_prompt, user_input=user_input))
print("-" * 40)

# Gemini 모델 실행
print("# Gemini Result:")
print(run_gemini_chain(system_prompt=system_prompt, user_input=user_input))
print("-" * 40)

파라미터 수정 예시를 알아봅시다.

In [ ]:
# 1. Temperature (창의성 조절)
print("# OpenAI (temperature=0.8)")
print(
    run_openai_chain(
        system_prompt=system_prompt, user_input=user_input, temperature=0.8
    )
)
print("-" * 40)


# 2. Top-p
print("# Claude (top_p=0.7)")
print(run_claude_chain(system_prompt=system_prompt, user_input=user_input, top_p=0.7))
print("-" * 40)


# 3. Max tokens (출력 길이 제한)
print("# Gemini (max_output_tokens=100)")
print(
    run_gemini_chain(
        system_prompt=system_prompt, user_input=user_input, max_output_tokens=100
    )
)
print("-" * 40)


# 4. 모델 이름 교체
print("# OpenAI (model='gpt-3.5-turbo')")
print(
    run_openai_chain(
        system_prompt=system_prompt, user_input=user_input, model="gpt-3.5-turbo"
    )
)
print("-" * 40)


# 5. 복수 파라미터 동시 변경
print("# Claude (temperature=0.9, top_p=0.95)")
print(
    run_claude_chain(
        system_prompt=system_prompt, user_input=user_input, temperature=0.9, top_p=0.95
    )
)
print("-" * 40)

#### 실습 확인을 위한 LangSmith 추적 세팅 함수

사용자가 실습 기록을 구분하기 위해 LangSmith 프로젝트명을 입력하면 되는 함수입니다.

입력한 이름으로 LangSmith 대시보드에 실행 내역이 저장됩니다.
(예: prompt-course, rag-lab1, myproject-001 등)


```python
# 프로젝트명을 변수로 바로 지정
LANGSMITH_PROJECT = "prompt-course"

# 함수 호출로 환경변수 등록
setup_langsmith(LANGSMITH_PROJECT)
```



In [ ]:
# LangSmith 설정 함수 (프로젝트명만 입력받아 환경변수 등록)


def setup_langsmith(project_name: str):
    """
    LangSmith 관련 환경변수를 등록하는 함수입니다.
    이미 등록된 LANGSMITH_API_KEY를 사용하며,
    project_name 변수로 LangSmith 프로젝트명을 지정할 수 있습니다.
    """
    LANGSMITH_ENDPOINT = "https://api.smith.langchain.com"
    LANGSMITH_TRACING = "true"

    os.environ.update(
        {
            "LANGSMITH_PROJECT": project_name,
            "LANGSMITH_ENDPOINT": LANGSMITH_ENDPOINT,
            "LANGSMITH_TRACING": LANGSMITH_TRACING,
        }
    )

    print("✅ LangSmith 설정 완료")
    print(f"- PROJECT : {project_name}")
    print(f"- ENDPOINT: {LANGSMITH_ENDPOINT}")
    print(f"- TRACING : {LANGSMITH_TRACING}")

#### 최종 실습 준비

In [ ]:
setup_langsmith("prompt-course")

### <font color='green'><b>[ 실습 ] </b></font> 싱글턴과 멀티턴 개념 정리

▶︎ **실습문제**: **"싱글턴"과 "멀티턴"의 기준에 대해 토론해보고 프롬프트를 제작해보세요.**

<hr>

(1) Prompt 제작 실습 :

  `싱글턴`과 `멀티턴`을 구분하는 프롬프트를 제작해보세요.

(2) 프롬프트 검증:

  테스트 데이터를 가지고 구분 기준에 따라 나오는지 테스트 해보세요.  (*데이터셋은 Discussion 2 사용)


<hr>

**테스트 데이터**: 📒 [사용자 프롬프트 데이터 모음](https://www.notion.so/b11833288f4247c19b157f430570c7d3?pvs=21)

<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂


```

</details>

✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
user_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(user_prompt)

✅ 이제 결과를 확인해봅시다!

In [ ]:
# OpenAI 결과 (gpt-4o-mini)
openai_response = run_openai_chain(user_input=user_prompt)
print(f"# OpenAI Result: {openai_response}")
print("-" * 50)

# Anthropic 결과 (claude-3-5-haiku-20241022)
anthropic_response = run_claude_chain(user_input=user_prompt)
print(f"# Anthropic Result: {anthropic_response}")
print("-" * 50)

# Gemini 결과 (gemini-2.0-flash)
gemini_response = run_gemini_chain(user_input=user_prompt)
print(f"# Gemini Result: {gemini_response}")

### <font color='green'><b>[ 실습 ] </b></font> 싱글턴과 멀티턴 Chunking 하기

▶︎ **실습문제**: **대화 데이터에서 ‘chit-chat’ 과 ‘non-chit chat’ 을 구분(chunking) 하는 프롬프트를 제작해보세요.**


<hr>

**대화 데이터**
```
<script>

Q: 확장앱
A: 크롬 확장앱은 아래 링크에서 다운받으실 수 있습니다!
크롬 확장앱을 사용하면 더욱 편리하게 인터넷을 이용하실 수 있습니다. 많은 분들이 사용하고 있는 유용한 확장앱도 많으니 한번쯤 살펴보시는 것을 추천드려요!😉 🔗 링크

Q: 정보보안 채용정보 찾아줘
A: 요청하신 정보보안 관련 채용 공고를 최신순으로 정렬한 결과입니다!

오케스트로 - DX Consultant (정보화전략 기획 및 수행)(근무지 : 서울)

# 클라우드 #정보화전략 #커뮤니케이션 #리서치 #문서작성

Q: 기획 관련 업무는 뭐가 있을까?
A: 기획 관련 업무는 다양한 분야에서 찾아볼 수 있습니다. 주요 업무들을 간단히 소개해 드리겠습니다:

전략 기획: 조직의 장기적인 목표와 방향 설정

프로젝트 기획: 특정 프로젝트의 목표, 일정, 예산 등을 계획

마케팅 기획: 제품이나 서비스의 홍보 전략 수립

상품 기획: 새로운 제품이나 서비스 개발 계획

이벤트 기획: 각종 행사나 프로모션 계획 및 실행

콘텐츠 기획: 미디어, 출판, 엔터테인먼트 등의 콘텐츠 개발

UX/UI 기획: 사용자 경험을 고려한 디지털 제품 설계

비즈니스 모델 기획: 새로운 사업 모델 개발 및 최적화

이 외에도 더 많은 분야가 있습니다. 특정 분야에 대해 더 자세히 알고 싶으시다면 말씀해 주세요.

Q: 채용 정보 찾을 수 있나? UX/UI 기획 현재 내 조건에서
A: 네, UX/UI 기획 분야에서 현재 귀하의 조건에 맞는 채용 정보를 찾을 수 있습니다. 다음과 같은 방법으로 접근해 보시는 것이 좋겠습니다:

온라인 구직 사이트 활용:

사람인, 잡코리아, 링크드인 등의 사이트에서 'UX/UI 기획' 키워드로 검색해보세요. 필터 기능을 사용하여 경력, 지역, 연봉 등 귀하의 조건에 맞게 설정하세요.

기업 홈페이지 확인:

관심 있는 회사의 채용 페이지를 직접 방문해 보세요. IT 기업이나 디자인 에이전시 등에서 자주 UX/UI 기획자를 채용합니다.

네트워킹:

UX/UI 관련 온라인 커뮤니티나 포럼에 참여해보세요. 관련 세미나나 컨퍼런스에 참석하여 업계 동향을 파악하고 인맥을 쌓아보세요.

포트폴리오 준비:

UX/UI 기획 분야에서는 실제 프로젝트 경험이 중요합니다. 개인 프로젝트나 학습 과정에서 만든 결과물로 포트폴리오를 구성해보세요.

채용 에이전시 활용:

UX/UI 분야 전문 헤드헌터나 채용 에이전시에 등록해보세요.

귀하의 구체적인 경력, 기술, 희망 조건 등에 대해 더 자세히 알려주시면, 더 구체적인 조언을 드릴 수 있습니다. 추가로 궁금한 점이 있으시면 말씀해 주세요.

Q: 이마트 집들이 선물 추천
A: 이마트에서 사면 좋을 집들이 선물을 추천해드리겠습니다.

과일 바구니: 고급스러운 포장으로 꾸며진 다양한 과일이 들어 있는 과일 바구니는 집들이에 인기 있는 선물입니다. 신혼부부가 새로 이사한 집에서 과일을 즐길 수 있다는 것이 좋은 포인트입니다.
와인 세트: 이마트에서 다양한 국산/수입 와인을 판매하고 있습니다. 레드, 화이트, 로제 등 다양한 종류의 와인을 골라 선물할 수 있으며, 와인잔이 포함된 와인 세트도 매력적입니다.
다양한 건강식품: 집들이 이벤트 선물로 건강에 좋은 다양한 건강식품을 추천합니다. 이마트에서는 건강에 좋은 음료, 다이어트 간식, 초콜릿 등을 판매하고 있습니다.
침구세트: 이마트에서는 다양한 침구세트를 판매하고 있습니다. 집들이 선물로 침구세트를 선물하는 것은 가족들이 안락한 휴식을 취하며 집안에서 행복한 시간을 보낼 수 있도록 돕는 좋은 아이디어입니다.
다양한 가전제품: 이마트에서는 다양한 종류의 가전제품을 판매하고 있습니다. 신혼부부들에게는 가전제품 선물도 매우 인기가 있으며, 전자레인지, 인덕션, 청소기 등 다양한 제품들을 선택할 수 있습니다.
꽃다발: 집들이에 꽃다발을 선물하는 것도 좋은 방법 중 하나입니다. 이마트에서는 다양한 종류의 꽃다발을 판매하고 있습니다. 축하의 마음을 전달하고 싶다면 꽃다발 선물을 추천합니다.위와 같이 다양한 제품들이 있으니, 신혼부부들이 좋아하는 것을 선택하셔서 선물해보세요.
Q: 우아한 거짓말 독후감 써봐
"우아한 거짓말"은 김려령 작가의 소설로, 학교 폭력과 그로 인한 비극적인 결과를 섬세하고 감동적으로 다루고 있습니다. 이 작품은 세 명의 여성 캐릭터, 천지, 혜지, 그리고 그들의 엄마가 중심이 되어 이야기가 펼쳐집니다. 천지는 어느 날 갑자기 자살을 하고, 그 이유는 가족에게조차 알려지지 않습니다. 이후 혜지와 그녀의 엄마는 천지의 죽음 뒤에 숨겨진 진실을 찾아가며 서로의 상처를 이해하고 치유하는 과정을 겪게 됩니다.
A: 작가는 인물들의 내면을 깊이 있게 그려내면서 독자로 하여금 각각의 캐릭터에 공감하게 만듭니다. 특히 학교 폭력의 심각성과 그 영향을 주변 인물들에게 미치는 파장을 현실적으로 묘사하며, 문제에 대한 사회적 인식을 높이는 데 기여합니다.

Q:충청남도 지역에 많은 비가 내려서 조합원들의 피해가 있을지도 몰라서 비 피해를 입은 조합원들은 조합으로 말씀해달라는 내용의 문자를 보내려고 하는데, 의미가 잘 전달될 수 있도록 문자를 작성해줘.
A: [충청남도 지역 조합 회원님께]
안녕하세요, 조합 회원님들께 알리고 싶은 사항이 있습니다. 최근 충청남도 지역에 폭우가 계속되어 농작물이나 시설물 등에 피해가 발생할 수 있습니다.
조합원님들 중에서 비로 인한 손실이나 피해를 겪으신 분들은 언제든지 조합으로 연락하여 주시기 바랍니다. 조합에서는 관련 상황을 확인한 후 최대한 도움을 드리겠습니다.
피해가 과장되지 않도록 주의를 기울이시고, 필요한 지원이 있으시면 조합으로 연락을 주시길 부탁드립니다.
감사합니다.
[충청남도 지역 조합]

</script>    
```
<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂


```

</details>

✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
system_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

In [ ]:
# 데이터셋 정의 -- 교재 실습자는 이 셀을 편집하지 마십시오.
data = """
<script>

Q: 확장앱
A: 크롬 확장앱은 아래 링크에서 다운받으실 수 있습니다!
크롬 확장앱을 사용하면 더욱 편리하게 인터넷을 이용하실 수 있습니다. 많은 분들이 사용하고 있는 유용한 확장앱도 많으니 한번쯤 살펴보시는 것을 추천드려요!😉 🔗 링크

Q: 정보보안 채용정보 찾아줘
A: 요청하신 정보보안 관련 채용 공고를 최신순으로 정렬한 결과입니다!

오케스트로 - DX Consultant (정보화전략 기획 및 수행)(근무지 : 서울)

# 클라우드 #정보화전략 #커뮤니케이션 #리서치 #문서작성

Q: 기획 관련 업무는 뭐가 있을까?
A: 기획 관련 업무는 다양한 분야에서 찾아볼 수 있습니다. 주요 업무들을 간단히 소개해 드리겠습니다:

전략 기획: 조직의 장기적인 목표와 방향 설정

프로젝트 기획: 특정 프로젝트의 목표, 일정, 예산 등을 계획

마케팅 기획: 제품이나 서비스의 홍보 전략 수립

상품 기획: 새로운 제품이나 서비스 개발 계획

이벤트 기획: 각종 행사나 프로모션 계획 및 실행

콘텐츠 기획: 미디어, 출판, 엔터테인먼트 등의 콘텐츠 개발

UX/UI 기획: 사용자 경험을 고려한 디지털 제품 설계

비즈니스 모델 기획: 새로운 사업 모델 개발 및 최적화

이 외에도 더 많은 분야가 있습니다. 특정 분야에 대해 더 자세히 알고 싶으시다면 말씀해 주세요.

Q: 채용 정보 찾을 수 있나? UX/UI 기획 현재 내 조건에서
A: 네, UX/UI 기획 분야에서 현재 귀하의 조건에 맞는 채용 정보를 찾을 수 있습니다. 다음과 같은 방법으로 접근해 보시는 것이 좋겠습니다:

온라인 구직 사이트 활용:

사람인, 잡코리아, 링크드인 등의 사이트에서 'UX/UI 기획' 키워드로 검색해보세요. 필터 기능을 사용하여 경력, 지역, 연봉 등 귀하의 조건에 맞게 설정하세요.

기업 홈페이지 확인:

관심 있는 회사의 채용 페이지를 직접 방문해 보세요. IT 기업이나 디자인 에이전시 등에서 자주 UX/UI 기획자를 채용합니다.

네트워킹:

UX/UI 관련 온라인 커뮤니티나 포럼에 참여해보세요. 관련 세미나나 컨퍼런스에 참석하여 업계 동향을 파악하고 인맥을 쌓아보세요.

포트폴리오 준비:

UX/UI 기획 분야에서는 실제 프로젝트 경험이 중요합니다. 개인 프로젝트나 학습 과정에서 만든 결과물로 포트폴리오를 구성해보세요.

채용 에이전시 활용:

UX/UI 분야 전문 헤드헌터나 채용 에이전시에 등록해보세요.

귀하의 구체적인 경력, 기술, 희망 조건 등에 대해 더 자세히 알려주시면, 더 구체적인 조언을 드릴 수 있습니다. 추가로 궁금한 점이 있으시면 말씀해 주세요.

Q: 이마트 집들이 선물 추천
A: 이마트에서 사면 좋을 집들이 선물을 추천해드리겠습니다.

과일 바구니: 고급스러운 포장으로 꾸며진 다양한 과일이 들어 있는 과일 바구니는 집들이에 인기 있는 선물입니다. 신혼부부가 새로 이사한 집에서 과일을 즐길 수 있다는 것이 좋은 포인트입니다.
와인 세트: 이마트에서 다양한 국산/수입 와인을 판매하고 있습니다. 레드, 화이트, 로제 등 다양한 종류의 와인을 골라 선물할 수 있으며, 와인잔이 포함된 와인 세트도 매력적입니다.
다양한 건강식품: 집들이 이벤트 선물로 건강에 좋은 다양한 건강식품을 추천합니다. 이마트에서는 건강에 좋은 음료, 다이어트 간식, 초콜릿 등을 판매하고 있습니다.
침구세트: 이마트에서는 다양한 침구세트를 판매하고 있습니다. 집들이 선물로 침구세트를 선물하는 것은 가족들이 안락한 휴식을 취하며 집안에서 행복한 시간을 보낼 수 있도록 돕는 좋은 아이디어입니다.
다양한 가전제품: 이마트에서는 다양한 종류의 가전제품을 판매하고 있습니다. 신혼부부들에게는 가전제품 선물도 매우 인기가 있으며, 전자레인지, 인덕션, 청소기 등 다양한 제품들을 선택할 수 있습니다.
꽃다발: 집들이에 꽃다발을 선물하는 것도 좋은 방법 중 하나입니다. 이마트에서는 다양한 종류의 꽃다발을 판매하고 있습니다. 축하의 마음을 전달하고 싶다면 꽃다발 선물을 추천합니다.위와 같이 다양한 제품들이 있으니, 신혼부부들이 좋아하는 것을 선택하셔서 선물해보세요.
Q: 우아한 거짓말 독후감 써봐
"우아한 거짓말"은 김려령 작가의 소설로, 학교 폭력과 그로 인한 비극적인 결과를 섬세하고 감동적으로 다루고 있습니다. 이 작품은 세 명의 여성 캐릭터, 천지, 혜지, 그리고 그들의 엄마가 중심이 되어 이야기가 펼쳐집니다. 천지는 어느 날 갑자기 자살을 하고, 그 이유는 가족에게조차 알려지지 않습니다. 이후 혜지와 그녀의 엄마는 천지의 죽음 뒤에 숨겨진 진실을 찾아가며 서로의 상처를 이해하고 치유하는 과정을 겪게 됩니다.
A: 작가는 인물들의 내면을 깊이 있게 그려내면서 독자로 하여금 각각의 캐릭터에 공감하게 만듭니다. 특히 학교 폭력의 심각성과 그 영향을 주변 인물들에게 미치는 파장을 현실적으로 묘사하며, 문제에 대한 사회적 인식을 높이는 데 기여합니다.

Q:충청남도 지역에 많은 비가 내려서 조합원들의 피해가 있을지도 몰라서 비 피해를 입은 조합원들은 조합으로 말씀해달라는 내용의 문자를 보내려고 하는데, 의미가 잘 전달될 수 있도록 문자를 작성해줘.
A: [충청남도 지역 조합 회원님께]
안녕하세요, 조합 회원님들께 알리고 싶은 사항이 있습니다. 최근 충청남도 지역에 폭우가 계속되어 농작물이나 시설물 등에 피해가 발생할 수 있습니다.
조합원님들 중에서 비로 인한 손실이나 피해를 겪으신 분들은 언제든지 조합으로 연락하여 주시기 바랍니다. 조합에서는 관련 상황을 확인한 후 최대한 도움을 드리겠습니다.
피해가 과장되지 않도록 주의를 기울이시고, 필요한 지원이 있으시면 조합으로 연락을 주시길 부탁드립니다.
감사합니다.
[충청남도 지역 조합]

</script>
"""

print("✔️ 데이터가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(system_prompt)

✅ 이제 결과를 확인해봅시다!

In [ ]:
# OpenAI 결과 (gpt-4o-mini)
openai_response = run_openai_chain(system_prompt=system_prompt, user_input=data)
print(f"# OpenAI Result: {openai_response}")
print("-" * 50)

# Anthropic 결과 (claude-3-5-haiku-20241022)
anthropic_response = run_claude_chain(system_prompt=system_prompt, user_input=data)
print(f"# Anthropic Result: {anthropic_response}")
print("-" * 50)

# Gemini 결과 (gemini-2.0-flash)
gemini_response = run_gemini_chain(system_prompt=system_prompt, user_input=data)
print(f"# Gemini Result: {gemini_response}")

### <font color='green'><b>[ 실습 ] </b></font> Response Analysis 1: Preference (Preferred vs Dispreferred Responses)

▶︎ **실습문제**: **다음 데이터 읽고 제작한 프롬프트로 사용자의 응답을 분류해보세요.**


<hr>

**Text Data**
```
사용자: 영어로 번역좀 해줘.  “안녕하세요 렌트비를 인상하는 것에 대해 우려가 많습니다. 작년에도 한 차례 렌트비 인상이 있었습니다. 당시 15% 인상률이었는데요. 이번 해에도 또 15%나 올리는 것은 부당하다고 생각됩니다. 저에게는 엄청난 경제적 부담이 돼요.

AI: Hello, I have many concerns about raising the rent. There was already an increase in rent last year. At that time, it was raised by 15%. I think it's unfair to raise it by another 15% this year. It's a huge financial burden for me.

사용자: 음.. 문장이 별로인데 좀 표현이 단호하면 좋겠어

```
<img src="images/02-Turn-Analysis-01.png" alt="Turn 분석하기 1번" width="800">

<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂
```

</details>

✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
system_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

In [ ]:
# 데이터셋 정의 -- 교재 실습자는 이 셀을 편집하지 마십시오.
data = """
<data>
사용자: 영어로 번역좀 해줘.  “안녕하세요 렌트비를 인상하는 것에 대해 우려가 많습니다. 작년에도 한 차례 렌트비 인상이 있었습니다. 당시 15% 인상률이었는데요. 이번 해에도 또 15%나 올리는 것은 부당하다고 생각됩니다. 저에게는 엄청난 경제적 부담이 돼요.

AI: Hello, I have many concerns about raising the rent. There was already an increase in rent last year. At that time, it was raised by 15%. I think it's unfair to raise it by another 15% this year. It's a huge financial burden for me.

사용자: 음.. 문장이 별로인데 좀 표현이 단호하면 좋겠어
</data>
"""

print("✔️ 데이터가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(system_prompt)

✅ 이제 결과를 확인해봅시다!

In [ ]:
# OpenAI 결과 (gpt-4o-mini)
openai_response = run_openai_chain(system_prompt=system_prompt, user_input=data)
print(f"# OpenAI Result: {openai_response}")
print("-" * 50)

# Anthropic 결과 (claude-3-5-haiku-20241022)
anthropic_response = run_claude_chain(system_prompt=system_prompt, user_input=data)
print(f"# Anthropic Result: {anthropic_response}")
print("-" * 50)

# Gemini 결과 (gemini-2.0-flash)
gemini_response = run_gemini_chain(system_prompt=system_prompt, user_input=data)
print(f"# Gemini Result: {gemini_response}")

### <font color='green'><b>[ 실습 ] </b></font>Response Analysis 2: Stance-Taking (Role Prompting, Persona Prompting, Multi-Persona Prompting)

▶︎ **실습문제**: **다음 데이터 읽고 제작한 프롬프트로 사용자의 응답을 분류해보세요.**


<hr>

**Text Data**
```
#데이터 사용자:

게임 좋아하는 아들이 게임 자제하는 법을 알려줘 AI: "게임을 즐기는 것은 좋지만, 게임 중독이나 과도한 게임 사용은 건강에 해로울 수 있습니다.
아들에게 게임 자제하는 방법을 알려주기 위해서는 몇 가지 팁이 있습니다. 1. 일정 시간 이상 플레이하지 않도록 규칙을 정해주세요.
2. 다른 취미나 운동 등 다양한 활동을 추천해주세요.
3. 아들과 함께 할 수 있는 가족 활동 계획 세우기를 제안하세요.
4. 온라인 친구와의 대화보다 현실에서 만날 수 있는 친구를 만드는 것도 도움이 됩니다. 위의 팁들은 아들에게 게임 자제하는 습관을 기르고, 더욱 건강하고 행복한 생활습관으로 이어질 수 있습니다."

사용자: 너무 교과서 적인데
```
<img src="images/02-Turn-Analysis-02.png" alt="Turn 분석하기 2번" width="800">
<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
# 정답 프롬프트는 패스트캠퍼스 강의 내에서만 확인하실 수 있습니다.
# 자세한 내용은 강의를 참고해주세요 🙂
```

</details>

✏️ 문제의 요구사항에 따라 프롬프트를 작성해봅시다!

In [ ]:
# ==============================================
# Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
system_prompt = """

여기에 작성할 prompt를 입력하세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역 ————

print("✔️ 프롬프트가 로드되었습니다!")

In [ ]:
# 데이터셋 정의 -- 교재 실습자는 이 셀을 편집하지 마십시오.
data = """
<data>
# 데이터 사용자:

게임 좋아하는 아들이 게임 자제하는 법을 알려줘 AI: "게임을 즐기는 것은 좋지만, 게임 중독이나 과도한 게임 사용은 건강에 해로울 수 있습니다.
아들에게 게임 자제하는 방법을 알려주기 위해서는 몇 가지 팁이 있습니다. 1. 일정 시간 이상 플레이하지 않도록 규칙을 정해주세요.
2. 다른 취미나 운동 등 다양한 활동을 추천해주세요.
3. 아들과 함께 할 수 있는 가족 활동 계획 세우기를 제안하세요.
4. 온라인 친구와의 대화보다 현실에서 만날 수 있는 친구를 만드는 것도 도움이 됩니다. 위의 팁들은 아들에게 게임 자제하는 습관을 기르고, 더욱 건강하고 행복한 생활습관으로 이어질 수 있습니다."

사용자: 너무 교과서 적인데
</data>
"""

print("✔️ 데이터가 로드되었습니다!")

☑️ 작성한 프롬프트를 확인해봅시다!

In [ ]:
print(system_prompt)

✅ 이제 결과를 확인해봅시다!

In [ ]:
# OpenAI 결과 (gpt-4o-mini)
openai_response = run_openai_chain(system_prompt=system_prompt, user_input=data)
print(f"# OpenAI Result: {openai_response}")
print("-" * 50)

# Anthropic 결과 (claude-3-5-haiku-20241022)
anthropic_response = run_claude_chain(system_prompt=system_prompt, user_input=data)
print(f"# Anthropic Result: {anthropic_response}")
print("-" * 50)

# Gemini 결과 (gemini-2.0-flash)
gemini_response = run_gemini_chain(system_prompt=system_prompt, user_input=data)
print(f"# Gemini Result: {gemini_response}")


<br>

<hr>